<a href="https://colab.research.google.com/github/dakshini01/ProdFusion/blob/main/codes/Bayesiyan_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymc

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_garment_data.csv to cleaned_garment_data (1).csv


In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 2. UPLOAD DATASET
# =========================

from google.colab import files
uploaded = files.upload()

Saving cleaned_garment_data.csv to cleaned_garment_data (2).csv


In [ ]:
# =========================
# 3.Load Dataset
# =========================
df = pd.read_csv("cleaned_garment_data.csv")

df.head()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   actual_productivity    1197 non-null   float64
 2   targeted_productivity  1197 non-null   float64
 3   smv                    1197 non-null   float64
 4   wip                    1197 non-null   float64
 5   over_time              1197 non-null   int64  
 6   incentive              1197 non-null   int64  
 7   idle_time              1197 non-null   float64
 8   idle_men               1197 non-null   int64  
 9   no_of_style_change     1197 non-null   int64  
 10  no_of_workers          1197 non-null   float64
 11  team                   1197 non-null   int64  
 12  wip_missing            1197 non-null   int64  
 13  department_sweing      1197 non-null   int64  
 14  day_Saturday           1197 non-null   int64  
 15  day_

In [ ]:
# =========================
# 4. Split DATASET
# =========================
df["date"] = pd.to_datetime(df["date"])

unique_dates = sorted(df["date"].unique())
split_index = int(0.8 * len(unique_dates))
cutoff_date = unique_dates[split_index]

train = df[df["date"] < cutoff_date]
test  = df[df["date"] >= cutoff_date]

# =========================
# 5. X and Y
# =========================

X_train = train.drop(columns=["actual_productivity", "date"])
y_train = train["actual_productivity"]

X_test = test.drop(columns=["actual_productivity", "date"])
y_test = test["actual_productivity"]

# =========================
# 6. Scale Dataset
# =========================

from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
    "team"
]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
# =========================
# 7. Build Bayesiyan
# =========================

import pymc as pm
import numpy as np

# Convert to numpy
X_train_np = X_train.values
y_train_np = y_train.values

n_features = X_train_np.shape[1]

with pm.Model() as model:

    # Priors for weights
    weights = pm.Normal("weights", mu=0, sigma=1, shape=n_features)

    # Bias term
    bias = pm.Normal("bias", mu=0, sigma=1)

    # Noise
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Linear model
    mu = pm.math.dot(X_train_np, weights) + bias

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train_np)

    # Sampling
    trace = pm.sample(1000, tune=1000, return_inferencedata=True)

Output()

In [ ]:
# =========================
# 8. Prediction
# =========================
# Posterior mean weights
weights_mean = trace.posterior["weights"].mean(dim=["chain","draw"]).values
bias_mean = trace.posterior["bias"].mean(dim=["chain","draw"]).values

# Predictions
y_pred_bayes = np.dot(X_test.values, weights_mean) + bias_mean